In [1]:
import rasterio
import numpy as np
import os
from rasterio.enums import ColorInterp
from rasterio.warp import transform_bounds


def get_raster_info(file_path: str, compute_stats: bool = True) -> dict:
    """
    Extract detailed raster metadata and statistics.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found")

    raster_info = {}

    with rasterio.open(file_path) as src:

        # --------------------------
        # 1️⃣ Basic Info
        # --------------------------
        raster_info["file_name"] = os.path.basename(file_path)
        raster_info["file_size_mb"] = round(os.path.getsize(file_path) / (1024 * 1024), 2)
        raster_info["driver"] = src.driver
        raster_info["width"] = src.width
        raster_info["height"] = src.height
        raster_info["band_count"] = src.count
        raster_info["dtypes"] = src.dtypes
        raster_info["nodata"] = src.nodata

        # --------------------------
        # 2️⃣ Spatial Info
        # --------------------------
        raster_info["crs"] = src.crs.to_string() if src.crs else None
        raster_info["bounds"] = src.bounds
        raster_info["resolution"] = src.res
        raster_info["transform"] = tuple(src.transform)

        if src.crs:
            raster_info["bounds_wgs84"] = transform_bounds(
                src.crs, "EPSG:4326", *src.bounds
            )
        else:
            raster_info["bounds_wgs84"] = None

        # --------------------------
        # 3️⃣ Advanced Info
        # --------------------------
        raster_info["compression"] = src.compression.value if src.compression else None
        raster_info["is_tiled"] = src.is_tiled
        raster_info["block_shapes"] = src.block_shapes
        raster_info["overviews"] = {
            i + 1: src.overviews(i + 1) for i in range(src.count)
        }

        # Check if likely COG
        raster_info["is_cog_like"] = (
            src.driver == "GTiff"
            and src.is_tiled
            and bool(src.overviews(1))
        )

        # --------------------------
        # 4️⃣ Band-Level Info
        # --------------------------
        bands_info = []

        for i in range(1, src.count + 1):
            band_data = {
                "band_number": i,
                "dtype": src.dtypes[i - 1],
                "color_interpretation": src.colorinterp[i - 1].name
                if src.colorinterp
                else None,
            }

            if compute_stats:
                band = src.read(i, masked=True)

                if np.ma.is_masked(band):
                    valid_data = band.compressed()
                else:
                    valid_data = band.flatten()

                if valid_data.size > 0:
                    band_data.update({
                        "min": float(valid_data.min()),
                        "max": float(valid_data.max()),
                        "mean": float(valid_data.mean()),
                        "std": float(valid_data.std()),
                    })
                else:
                    band_data.update({
                        "min": None,
                        "max": None,
                        "mean": None,
                        "std": None,
                    })

            bands_info.append(band_data)

        raster_info["bands"] = bands_info

        # --------------------------
        # 5️⃣ Extra Metadata Tags
        # --------------------------
        raster_info["tags"] = src.tags()
        raster_info["band_tags"] = {
            i + 1: src.tags(i + 1) for i in range(src.count)
        }

    return raster_info

In [4]:
reps=get_raster_info(file_path=f"fast_backend/media/Rajat_data/shape_stp/STP_pripority_raster/Visually/Downstream_Effect_of_Drain.tif")

In [5]:
print(reps)

{'file_name': 'Downstream_Effect_of_Drain.tif', 'file_size_mb': 35.02, 'driver': 'GTiff', 'width': 4452, 'height': 1983, 'band_count': 1, 'dtypes': ('float32',), 'nodata': -3.4028230607370965e+38, 'crs': 'EPSG:32644', 'bounds': BoundingBox(left=573425.873405851, bottom=2793508.247357674, right=706985.873405851, top=2852998.247357674), 'resolution': (30.0, 30.0), 'transform': (30.0, 0.0, 573425.873405851, 0.0, -30.0, 2852998.247357674, 0.0, 0.0, 1.0), 'bounds_wgs84': (81.72914556565279, 25.243628104218697, 83.06415145270826, 25.793305337205396), 'compression': None, 'is_tiled': True, 'block_shapes': [(128, 128)], 'overviews': {1: []}, 'is_cog_like': False, 'bands': [{'band_number': 1, 'dtype': 'float32', 'color_interpretation': 'gray', 'min': 3.0, 'max': 5.0, 'mean': 4.503371238708496, 'std': 0.6449609398841858}], 'tags': {'DataType': 'Generic', 'AREA_OR_POINT': 'Area'}, 'band_tags': {1: {'RepresentationType': 'ATHEMATIC', 'STATISTICS_COVARIANCES': '0.4159747423363741', 'STATISTICS_MAXI